# Kymora: High-Throughput Time-Series Feature Extraction
### Fast, SIMD-Accelerated Time-Series Feature Extraction with TSFEL Drop-In Parity

This tutorial demonstrates:
1. **Real-World Dataset Ingestion** using `kymora.datasets`.
2. **Native Rust Feature Extraction** with profiles (`core33`, `extended`, `full`).
3. **TSFEL Drop-In Compatibility** (`from kymora.compat import tsfel`) for up to 1,000x faster execution.
4. **Cepstral (MFCC/LPCC) & Wavelet (CWT)** feature extraction on real patient biosignals.
5. **Unsupervised Collinear Feature Pruning** via `kymora.utils.correlated_features`.
6. **Downstream Classification** using scikit-learn.


In [ ]:
import time
import numpy as np
import pandas as pd
import kymora
from kymora.datasets import load_dataset
from kymora.utils import correlated_features
from kymora.compat import tsfel
print(f'Kymora version: {kymora.__version__}')

## 1. Load Real-World Dataset
We load real electrocardiogram (ECG) heartbeats from the UCR Time Series Archive.

In [ ]:
X_train, y_train, X_test, y_test = load_dataset('ECG200')
print(f'Training heartbeats: {X_train.shape}')
print(f'Test heartbeats:     {X_test.shape}')
print(f'Cardiac cycle length: {X_train.shape[1]} timestamps')

## 2. High-Throughput Native Extraction
Kymora borrows buffers directly from NumPy, releases the Python GIL, and uses Rayon multi-threading.

In [ ]:
t0 = time.perf_counter()
df_core = kymora.extract_features_df(X_train, profile='core33')
t_el = (time.perf_counter() - t0) * 1000.0
print(f'Extracted {df_core.shape[1]} features for {len(X_train)} series in {t_el:.2f} ms!')
df_core.head(3)

## 3. Zero-Code TSFEL Drop-In Migration
Replace `import tsfel` with `from kymora.compat import tsfel`. The rest of your code remains 100% identical.

In [ ]:
# Exact same TSFEL API syntax:
cfg = tsfel.get_features_by_domain('statistical')
features_df = tsfel.time_series_features_extractor(cfg, X_train, fs=100.0)
print(f'Extracted shape: {features_df.shape}')

# Correlation filtering:
clean_df = tsfel.correlated_features(features_df, threshold=0.90)
print(f'Shape after correlation pruning: {clean_df.shape}')

## 4. Specialized Cepstral & Continuous Wavelet Transform (CWT)
Kymora supports 12 Mel-Frequency Cepstral Coefficients (MFCC), 12 Linear Prediction Cepstral Coefficients (LPCC), and 37 Continuous Wavelet Transform (CWT) features.

In [ ]:
special_features = [
    'mean', 'std', 'root_mean_square',
    'mfcc_0', 'mfcc_1', 'mfcc_2',
    'lpcc_0', 'lpcc_1', 'lpcc_2',
    'wavelet_energy_scale_1', 'wavelet_energy_scale_2',
    'wavelet_entropy'
]
df_special = kymora.extract_features_df(X_train[:5], features=special_features, fs=100.0)
df_special

## 5. Downstream Machine Learning Model Evaluation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df_tr = kymora.extract_features_df(X_train, profile='core33')
df_te = kymora.extract_features_df(X_test, profile='core33')

df_tr_clean = correlated_features(df_tr, threshold=0.90)
cols = list(df_tr_clean.columns)
df_te_clean = df_te[cols]

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(df_tr_clean)
X_te_s = scaler.transform(df_te_clean)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_tr_s, y_train)
preds = clf.predict(X_te_s)
print(f'Random Forest Test Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%')
print(classification_report(y_test, preds, digits=3))